# 04. Comparação descritiva e análise por regime

Roteiro da Etapa 2. **Este notebook só é executado depois de `python -m src.pipeline.final_test`**: ele lê `results/test.csv`, gerado uma única vez após o congelamento do protocolo (`results/protocolo.lock`). Nada do que aparece aqui pode retroalimentar o modelo, o espaço de busca ou a seleção de datasets.

O notebook usa as funções que o grupo implementa em `src/pipeline/descriptive.py`, `src/pipeline/regime.py` e `src/reports/`. Enquanto uma função não estiver implementada, a célula correspondente falha com `NotImplementedError`; os testes em `tests/aceitacao/` indicam quando cada função está correta.

Roteiro da comparação (puramente descritiva, sem testes de hipótese):
1. uma medida por dataset: a média dos 3 folds externos oficiais (folds não são observações independentes);
2. tabelas de médias e desvios por sistema e por dataset;
3. postos por dataset (1 = melhor) e posto médio de cada sistema;
4. vitórias, empates e derrotas do modelo do grupo (com busca) contra cada baseline;
5. padrão vs. com busca de hiperparâmetros (ablação), dataset a dataset;
6. regimes e custo computacional.

As respostas às perguntas devem constar no relatório.

In [ ]:
import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

from src.pipeline.descriptive import (
    ablation_summary,
    ablation_table,
    mean_rank_per_dataset,
    wins_ties_losses,
)
from src.pipeline.evaluate import HIGHER_IS_BETTER
from src.pipeline.regime import REGIME_COLUMNS, aggregate_by_regime, rank_by_regime, regime_metadata
from src.pipeline.results import load_raw, per_dataset
from src.reports.plots import boxplot_by_model, cost_vs_performance
from src.reports.results_table import per_dataset_table, summary_by_model

RESULTS = REPO / 'results'
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
METRIC = 'auc_ovo'          # métrica primária (declarada antes da análise)
REFERENCE = 'group_tuned'   # referência: modelo do grupo após a busca
SYSTEMS = [REFERENCE, 'lightgbm', 'xgboost', 'catboost']
HIB = HIGHER_IS_BETTER[METRIC]

In [ ]:
raw = load_raw(RESULTS / 'test.csv')   # avaliação final única (final_test)
print(raw['status'].value_counts())
raw[raw['status'] != 'ok'][['dataset', 'fold', 'model', 'error']]

## 1. Tabelas descritivas

In [ ]:
summary_by_model(raw)

In [ ]:
per_dataset_table(raw, METRIC)

In [ ]:
table = per_dataset(raw, METRIC, models=SYSTEMS)   # uma medida por dataset (média dos folds)
N, k = table.shape
print(f'N = {N} datasets completos, k = {k} sistemas')
table

**Pergunta 1.** Quantos datasets ficaram de fora pela regra de completude e por quê? Por que a unidade de comparação é o dataset (média dos 3 folds externos) e não cada fold?

## 2. Postos por dataset

In [ ]:
mean_rank_per_dataset(table, higher_is_better=HIB)

**Postos por dataset (figura do relatório, item c).** A partir de `table`, calcule o posto de cada sistema em cada dataset (1 = melhor, empates com posto médio, respeitando `HIB`) e salve um mapa de calor em `FIGURES / 'postos_por_dataset.png'`.

**Pergunta 2.** Qual sistema tem o menor posto médio? A ordem dos postos médios coincide com a ordem das médias da métrica (seção 1)? Se não, o que explica a diferença (considere a escala da métrica em cada dataset)?

## 3. Vitórias, empates e derrotas do modelo do grupo

In [ ]:
wins_ties_losses(table, REFERENCE, higher_is_better=HIB)

In [ ]:
# Sensibilidade à tolerância: diferenças muito pequenas contam como empate.
wins_ties_losses(table, REFERENCE, higher_is_better=HIB, tolerance=0.005)

**Pergunta 3.** Contra quais baselines o modelo do grupo vence na maioria dos datasets? As contagens mudam muito com a tolerância? Compare as diferenças médias (`mean_diff`) com o desvio entre folds de cada dataset (seção 1).

## 4. Padrão vs. com busca (ablação)

**Nota.** A comparação entre `group_tuned` e `group_default` sobre o teste é **descritiva** do resultado final: ela não serve para decidir se a busca deve ser usada nem para alterar o espaço de busca. Os sistemas e a referência `group_tuned` foram fixados a priori pelo enunciado; `results/validation.csv` é apenas diagnóstico (o treino de um fold externo contém o teste dos outros) e não foi base dessa escolha.

In [ ]:
ablation = ablation_table(raw, METRIC, higher_is_better=HIB)
ablation

In [ ]:
ablation_summary(ablation)

**Pergunta 4.** Em quantos datasets a busca de hiperparâmetros melhorou o modelo no teste? O ganho médio compensa o tempo de busca? O resultado confirma o que a validação interna indicava?

## 5. Regimes e custo computacional

In [ ]:
meta = regime_metadata(table.index.tolist())
for col in REGIME_COLUMNS:
    print(f'\n=== {col} ===')
    print(aggregate_by_regime(table, meta, col).to_string(index=False))
rank_by_regime(table, meta, 'regime_size', higher_is_better=HIB)

In [ ]:
boxplot_by_model(table, METRIC, output=FIGURES / f'boxplot_{METRIC}.png');

In [ ]:
cost_vs_performance(raw, METRIC, output=FIGURES / 'custo_vs_desempenho.png');

**Pergunta 5.** Em quais regimes o modelo do grupo se destaca ou falha? Por que essa análise exige cautela (considere o número de datasets por célula, 4 por regime de tamanho)? Qual sistema oferece a melhor relação entre custo e desempenho?

**Pergunta 6.** Repita as seções 2 a 4 para uma métrica secundária (por exemplo, entropia cruzada, em que menor é melhor). Os postos e as contagens de vitórias são estáveis quando se troca a métrica?

**Pergunta 7 (conceitual).** Que conclusões sobre o modelo tema podem ser sustentadas com 12 datasets sem teste de hipótese, e quais não podem? Considere a magnitude das diferenças em relação ao desvio entre folds, a estabilidade dos postos e das contagens de vitórias quando se troca a métrica, o pequeno número de datasets por regime (4) e o risco de generalizar a partir de poucas bases.